In [ ]:
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import seaborn as sns
import pandas as pd
import numpy as np
import yfinance as yf
from typing import Optional
import requests
from datetime import datetime, timedelta


In [ ]:
stocks_name = ['BTC-USD', 'ETH-USD', 'SPY', 'GLD', '^VIX']

In [ ]:
# Verificar data atual e forçar atualização
print(f"Data atual: {pd.Timestamp.now().date()}")
print("Forçando download dos dados...")

# Usar end='2026-04-10' para garantir que pegue dados mais recentes
stocks_data = yf.download(stocks_name, start='2018-01-01', end=(datetime.now() + timedelta(days=2)).strftime('%Y-%m-%d'), interval='1wk', auto_adjust=True, prepost=False)

print(f"Período dos dados: {stocks_data.index[0].date()} a {stocks_data.index[-1].date()}")
print(f"Última semana disponível: {stocks_data.index[-1].date()}")

Data atual: 2026-04-10
Forçando download dos dados...


[*********************100%***********************]  5 of 5 completed

Período dos dados: 2018-01-01 a 2026-04-06
Última semana disponível: 2026-04-06


In [ ]:
df_stocks = pd.DataFrame(stocks_data).dropna().copy()

In [ ]:
df_stocks_close = df_stocks['Close'].dropna().drop(columns=['^VIX']).copy()

Normalized BTC/USD, SPY and GLD

In [ ]:
normalized_df = (df_stocks_close / df_stocks_close.iloc[0])

In [ ]:
fig = make_subplots(
    rows=1,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    row_heights=[1.5]
)

fig.add_trace(go.Scatter(
    x=normalized_df.index,
    y=normalized_df['BTC-USD']
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=normalized_df.index,
    y=normalized_df['SPY']
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=normalized_df.index,
    y=normalized_df['GLD']
), row=1, col=1)

fig.update_layout(
    title='Normalized Prices (BTC, SPY, GLD)',
    xaxis_title='Date',
    yaxis_title='Normalized Price',
    height=600,
    showlegend=True,
    template='plotly_dark'
)

fig.show()

Copy the DataFrame to a new variable

In [ ]:
df_stocks_close_criptos = df_stocks['Close'].copy()

Thechnical Indicators

In [ ]:
###
# Calculate SMAs for BTC-USD
df_stocks_close_criptos['BTC-USD_EMA-50'] = df_stocks_close_criptos['BTC-USD'].ewm(span=50).mean()
df_stocks_close_criptos['BTC-USD_EMA-100'] = df_stocks_close_criptos['BTC-USD'].ewm(span=100).mean()
df_stocks_close_criptos['BTC-USD_EMA-200'] = df_stocks_close_criptos['BTC-USD'].ewm(span=200).mean()
###
# Calculate SMAs for ETH-USD
df_stocks_close_criptos['ETH-USD_EMA-50'] = df_stocks_close_criptos['ETH-USD'].ewm(span=50).mean()
df_stocks_close_criptos['ETH-USD_EMA-100'] = df_stocks_close_criptos['ETH-USD'].ewm(span=100).mean()
df_stocks_close_criptos['ETH-USD_EMA-200'] = df_stocks_close_criptos['ETH-USD'].ewm(span=200).mean()
###

MACD

In [ ]:
###
#MACD
def calculate_macd(df, fast=12, slow=26, signal=9):
    exp1 = df_stocks['Close']['BTC-USD'].ewm(span=fast).mean()
    exp2 = df_stocks['Close']['BTC-USD'].ewm(span=slow).mean()
    macd = exp1 - exp2
    signal = macd.ewm(span=signal).mean()
    histogram = macd - signal
    return macd, signal, histogram


macd, signal, histogram = calculate_macd(df_stocks)

In [ ]:
###
#MACD SPY
def calculate_macd_spy(df, fast=12, slow=26, signal=9):
    exp1 = df_stocks['Close']['SPY'].ewm(span=fast).mean()
    exp2 = df_stocks['Close']['SPY'].ewm(span=slow).mean()
    macd = exp1 - exp2
    signal = macd.ewm(span=signal).mean()
    histogram = macd - signal
    return macd, signal, histogram


macd_spy, signal_spy, histogram_spy = calculate_macd_spy(df_stocks)

BB %B - Bollinger Bands Percentage Bandwidth

In [ ]:
window = 20
std_dev = 2

price = df_stocks["Close"]["BTC-USD"]
middle = price.rolling(window).mean()
std = price.rolling(window).std()
upper = middle + std_dev * std
lower = middle - std_dev * std

bb_percent_b = (price - lower) / (upper - lower)

In [ ]:
window_spy = 20
std_dev_spy = 2

price_spy = df_stocks["Close"]["SPY"]
middle_spy = price_spy.rolling(window_spy).mean()
std_spy = price_spy.rolling(window_spy).std()
upper_spy = middle_spy + std_dev_spy * std_spy
lower_spy = middle_spy - std_dev_spy * std_spy

bb_percent_spy = (price_spy - lower_spy) / (upper_spy - lower_spy)

Stochastic + RSI

In [ ]:
# RSI
def calculate_rsi(prices, window=14):
    delta = prices.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

# Stochastic Oscillator
def calculate_stochastic(high, low, close, k_window=14, d_window=3):
    lowest_low = low.rolling(window=k_window).min()
    highest_high = high.rolling(window=k_window).max()
    k_percent = 100 * ((close - lowest_low) / (highest_high - lowest_low))
    d_percent = k_percent.rolling(window=d_window).mean()
    return k_percent, d_percent

# Calcular para BTC-USD
rsi_btc = calculate_rsi(df_stocks['Close']['BTC-USD'])
stoch_k_btc, stoch_d_btc = calculate_stochastic(
    df_stocks['High']['BTC-USD'],
    df_stocks['Low']['BTC-USD'],
    df_stocks['Close']['BTC-USD']
)

In [ ]:
# RSI
def calculate_rsi(prices, window=14):
    delta = prices.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

# Stochastic Oscillator
def calculate_stochastic(high, low, close, k_window=14, d_window=3):
    lowest_low = low.rolling(window=k_window).min()
    highest_high = high.rolling(window=k_window).max()
    k_percent = 100 * ((close - lowest_low) / (highest_high - lowest_low))
    d_percent = k_percent.rolling(window=d_window).mean()
    return k_percent, d_percent

# Calcular para SPY
rsi_spy = calculate_rsi(df_stocks['Close']['SPY'])
stoch_k_spy, stoch_d_spy = calculate_stochastic(
    df_stocks['High']['SPY'],
    df_stocks['Low']['SPY'],
    df_stocks['Close']['SPY']
)

Returns of the investments in multiplication

In [ ]:
stocks_returns_week_df = (df_stocks_close.pct_change().dropna())#Day variation
acumulated_returns = (1 + stocks_returns_week_df).cumprod()

Log Return

In [ ]:
stocks_returns_log_df = np.log(df_stocks_close/ df_stocks_close.shift(1)).dropna()

Volatility Weeks Sharpe & Sortino

In [ ]:
trading_weeks_sharpe = 60#30
trading_weeks_sortino = 60
# Volatilidade / janelas (dados mensais)
# Observação: a anualização para mensal deve usar 12 (meses por ano),
# independente do tamanho da janela usada no rolling.

periods_per_week = 52

# Volatilidade anualizada (usada no Sharpe)
volatility_week_sharpe = (
    stocks_returns_log_df.rolling(window=trading_weeks_sharpe).std() * np.sqrt(periods_per_week)
)

# (opcional) Mantém a variável para consistência com o resto do notebook
volatility_week_sortino = (
    stocks_returns_log_df.rolling(window=trading_weeks_sortino).std() * np.sqrt(periods_per_week)
)

Sharpe Ratio Weeks(52)

In [ ]:
rf = 0.01

rolling_mean = stocks_returns_log_df.rolling(window=trading_weeks_sharpe).mean() * periods_per_week
rolling_std = stocks_returns_log_df.rolling(window=trading_weeks_sharpe).std() * np.sqrt(periods_per_week)

sharpe_ratio = (rolling_mean - rf) / rolling_std


sharpe_ratio = sharpe_ratio.replace([np.inf, -np.inf], np.nan)

Sortino Ratio

In [ ]:
stocks_returns_week_df

Ticker,BTC-USD,ETH-USD,GLD,SPY
Date,,,,
2018-01-08,-0.164199,0.185229,0.013006,0.016458
2018-01-15,-0.157704,-0.232073,-0.004253,0.008959
2018-01-22,0.016052,0.187151,0.013052,0.022004
2018-01-29,-0.297743,-0.330116,-0.013118,-0.038837
2018-02-05,-0.017765,-0.023988,-0.012817,-0.050644
...,...,...,...,...
2026-03-09,0.103383,0.124378,-0.026758,-0.015006
2026-03-16,-0.067931,-0.057143,-0.102986,-0.020716
2026-03-23,-0.027862,-0.034334,0.003193,-0.019656


In [ ]:
rf = 0.01
mar = rf / periods_per_week

returns = stocks_returns_week_df

# excesso de retorno
excess = returns - mar

# downside (apenas perdas relativas ao MAR)
downside = np.minimum(0, excess)

# cálculo
downside_deviation_week = downside.rolling(window=trading_weeks_sortino).std() * np.sqrt(periods_per_week)

rolling_mean_week = excess.rolling(window=trading_weeks_sortino).mean() * periods_per_week

sortino_ratio = rolling_mean_week / downside_deviation_week

In [ ]:
sharpe_ratio

Ticker,BTC-USD,ETH-USD,GLD,SPY
Date,,,,
2018-01-08,NaN,NaN,NaN,NaN
2018-01-15,NaN,NaN,NaN,NaN
2018-01-22,NaN,NaN,NaN,NaN
2018-01-29,NaN,NaN,NaN,NaN
2018-02-05,NaN,NaN,NaN,NaN
...,...,...,...,...
2026-03-09,-0.758342,-0.476950,2.762645,0.547180
2026-03-16,-0.940200,-0.556162,1.859531,0.342817
2026-03-23,-0.896115,-0.458809,1.827108,0.290074


In [ ]:
sortino_ratio

Ticker,BTC-USD,ETH-USD,GLD,SPY
Date,,,,
2018-01-08,NaN,NaN,NaN,NaN
2018-01-15,NaN,NaN,NaN,NaN
2018-01-22,NaN,NaN,NaN,NaN
2018-01-29,NaN,NaN,NaN,NaN
2018-02-05,NaN,NaN,NaN,NaN
...,...,...,...,...
2026-03-09,-0.933302,-0.212707,6.382606,1.029402
2026-03-16,-1.226423,-0.361546,3.491284,0.692962
2026-03-23,-1.156541,-0.185901,3.433879,0.606396


Data Conditioning of BTC and SPY

In [ ]:
# %% [markdown]
# ## Data Conditioning BTC

nv_shp_btc_top = 2.05
nv_srt_btc_top = 5.65
nv_shp_btc_bottom = -1.1
nv_srt_btc_bottom = -1.7
mask_btc_sortino_top = sortino_ratio['BTC-USD'] > nv_srt_btc_top
mask_btc_sharpe_top = sharpe_ratio['BTC-USD'] > nv_shp_btc_top
mask_btc_sharpe_sortino_top = (sortino_ratio['BTC-USD'] >= nv_srt_btc_top) & (sharpe_ratio['BTC-USD'] > nv_shp_btc_top)
mask_btc_sortino_reversion_down = (sharpe_ratio['BTC-USD'] < 1.2) & (sortino_ratio['BTC-USD'] >= nv_srt_btc_top)
mask_btc_sharpe_bottom = sharpe_ratio['BTC-USD'] < nv_shp_btc_bottom
mask_btc_sortino_bottom = sortino_ratio['BTC-USD'] < nv_srt_btc_bottom
mask_btc_sharpe_sortino_bottom = (sortino_ratio['BTC-USD'] < nv_srt_btc_bottom) & (sharpe_ratio['BTC-USD'] < nv_shp_btc_bottom)
mask_btc_sortino_reversion_up = (sharpe_ratio['BTC-USD'] > 0) & (sortino_ratio['BTC-USD'] <= nv_srt_btc_bottom)

# %% [markdown]
# ## Data Conditioning SPY
# %%

nv_shp_spy_top = 2.2
nv_srt_spy_top = 5
nv_shp_spy_bottom = -0.5
nv_srt_spy_bottom = -1.1
mask_spy_sortino_top = sortino_ratio['SPY'] >= nv_srt_spy_top
mask_spy_sharpe_top = sharpe_ratio['SPY'] > nv_shp_spy_top
mask_spy_sharpe_sortino_top = (sortino_ratio['SPY'] >= nv_srt_spy_top) & (sharpe_ratio['SPY'] > nv_shp_spy_top)
mask_spy_sortino_reversion_down = (sortino_ratio['SPY'] >= 3.0) & (sharpe_ratio['SPY'] < nv_shp_spy_bottom)
mask_spy_sharpe_bottom = sharpe_ratio['SPY'] < nv_shp_spy_bottom
mask_spy_sortino_bottom = (sortino_ratio['SPY'] < nv_srt_spy_bottom)
mask_spy_sharpe_sortino_bottom = (sortino_ratio['SPY'] < nv_srt_spy_bottom) & (sharpe_ratio['SPY'] < nv_shp_spy_bottom)
mask_spy_sortino_reversion_up = (sharpe_ratio['SPY'] > 1.0) & (sortino_ratio['SPY'] < 0)


In [ ]:
# -----------------------------
# Suavização dos indicadores
# -----------------------------
sharpe_smooth = sharpe_ratio.rolling(1, min_periods=1).mean()
sortino_smooth = sortino_ratio.rolling(1, min_periods=1).mean()

# -----------------------------
# Função genérica de sinais
# -----------------------------
def get_signals_top(asset, close, bb, rsi, sharpe_thresh, sortino_thresh, bb_thresh, rsi_thresh):
    points_top = []
    points_top_confirmed = []

    # tamanho seguro para todas as séries
    length = min(len(close), len(sharpe_smooth[asset]), len(sortino_smooth[asset]), len(bb), len(rsi))

    for i in range(1, length):
        # sinal topo
        top = (sharpe_smooth[asset].iloc[i] > sharpe_thresh) & \
              (sortino_smooth[asset].iloc[i] > sortino_thresh) & \
              (bb.iloc[i] > bb_thresh) & \
              (rsi.iloc[i] > rsi_thresh)
        if top:
            points_top.append(i)
        # confirmação
        top_confirmed = top & \
                        (sharpe_smooth[asset].iloc[i-1] > sharpe_thresh) & \
                        (sortino_smooth[asset].iloc[i-1] > sortino_thresh) & \
                        (bb.iloc[i-1] > bb_thresh) & \
                        (rsi.iloc[i-1] > rsi_thresh)
        if top_confirmed:
            points_top_confirmed.append(i)

    return points_top, points_top_confirmed

def get_signals_bottom(asset, close, bb, rsi, sharpe_thresh, sortino_thresh, bb_thresh, rsi_thresh):
    points_bottom = []
    points_bottom_confirmed = []

    # tamanho seguro para todas as séries
    length = min(len(close), len(sharpe_smooth[asset]), len(sortino_smooth[asset]), len(bb), len(rsi))

    for i in range(1, length):
        # sinal fundo
        bottom = (sharpe_smooth[asset].iloc[i] < sharpe_thresh) & \
                 (sortino_smooth[asset].iloc[i] < sortino_thresh) & \
                 (bb.iloc[i] < bb_thresh) & \
                 (rsi.iloc[i] < rsi_thresh)
        if bottom:
            points_bottom.append(i)
        # confirmação
        bottom_confirmed = bottom & \
                           (sharpe_smooth[asset].iloc[i-1] < sharpe_thresh) & \
                           (sortino_smooth[asset].iloc[i-1] < sortino_thresh) & \
                           (bb.iloc[i-1] < bb_thresh) & \
                           (rsi.iloc[i-1] < rsi_thresh)
        if bottom_confirmed:
            points_bottom_confirmed.append(i)

    return points_bottom, points_bottom_confirmed

# -----------------------------
# Aplicando para BTC e SPY
# -----------------------------

top_btc, top_btc_confirmed = get_signals_top(
    'BTC-USD', df_stocks['Close'], bb_percent_b, rsi_btc,
    sharpe_thresh=nv_shp_btc_top, sortino_thresh=nv_srt_btc_top, bb_thresh=0.9, rsi_thresh=80
)

top_spy, top_spy_confirmed = get_signals_top(
    'SPY', df_stocks['Close'], bb_percent_spy, rsi_spy,
    sharpe_thresh=nv_shp_spy_top, sortino_thresh=nv_srt_spy_top, bb_thresh=0.8, rsi_thresh=70
)

bottom_btc, bottom_btc_confirmed = get_signals_bottom(
    'BTC-USD', df_stocks['Close'], bb_percent_b, rsi_btc,
    sharpe_thresh=nv_shp_btc_bottom, sortino_thresh=nv_srt_btc_bottom, bb_thresh=0.25, rsi_thresh=28
)

bottom_spy, bottom_spy_confirmed = get_signals_bottom(
    'SPY', df_stocks['Close'], bb_percent_spy, rsi_spy,
    sharpe_thresh=nv_shp_spy_bottom, sortino_thresh=nv_srt_spy_bottom, bb_thresh=0.25, rsi_thresh=28
)

print("Top BTC:", len(top_btc))
print("Top BTC Confirmed:", len(top_btc_confirmed))
print("Top SPY:", len(top_spy))
print("Top SPY Confirmed:", len(top_spy_confirmed))
print("Bottom BTC:", len(bottom_btc))
print("Bottom BTC Confirmed:", len(bottom_btc_confirmed))
print("Bottom SPY:", len(bottom_spy))
print("Bottom SPY Confirmed:", len(bottom_spy_confirmed))

Top BTC: 3
Top BTC Confirmed: 1
Top SPY: 4
Top SPY Confirmed: 1
Bottom BTC: 2
Bottom BTC Confirmed: 0
Bottom SPY: 0
Bottom SPY Confirmed: 0


In [ ]:
def macro_regime_spy(mask_top, mask_bottom):
    macro_points = [0]
    current_regime = 0

    for i in range(len(mask_top)):
        if mask_top.iloc[i] and not mask_bottom.iloc[i]:
            current_regime = 1
        elif mask_bottom.iloc[i] and not mask_top.iloc[i]:
            current_regime = -1


        macro_points.append(current_regime)

    return macro_points

macro_result_spy = macro_regime_spy(mask_spy_sharpe_sortino_top, mask_spy_sharpe_sortino_bottom)


In [ ]:
def macro_regime_btc(mask_top, mask_bottom):
    macro_points = [0]
    current_regime = 0

    for i in range(len(mask_top)):
        if mask_top.iloc[i] and not mask_bottom.iloc[i]:
            current_regime = 1
        elif mask_bottom.iloc[i] and not mask_top.iloc[i]:
            current_regime = -1


        macro_points.append(current_regime)

    return macro_points

macro_result_btc = macro_regime_btc(mask_btc_sharpe_sortino_top, mask_btc_sharpe_sortino_bottom)


In [ ]:
date_result_spy = df_stocks['Close']['SPY'].index

In [ ]:
macro_result_df = pd.DataFrame(index=date_result_spy, data={'BTC_Macro': macro_result_btc, 'SPY_Macro': macro_result_spy})

In [ ]:
sortino_ratio


Ticker,BTC-USD,ETH-USD,GLD,SPY
Date,,,,
2018-01-08,NaN,NaN,NaN,NaN
2018-01-15,NaN,NaN,NaN,NaN
2018-01-22,NaN,NaN,NaN,NaN
2018-01-29,NaN,NaN,NaN,NaN
2018-02-05,NaN,NaN,NaN,NaN
...,...,...,...,...
2026-03-09,-0.933302,-0.212707,6.382606,1.029402
2026-03-16,-1.226423,-0.361546,3.491284,0.692962
2026-03-23,-1.156541,-0.185901,3.433879,0.606396


In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=macro_result_df.index,
    y=macro_result_df['BTC_Macro'],
    mode='lines+markers',
    line_shape='hv',
    name='BTC Macro Regime',
    line=dict(color='orange', width=2),
    marker=dict(color=['red' if m == -1 else 'green' if m == 1 else 'gray' for m in macro_result_df['BTC_Macro']])
))

fig.add_trace(go.Scatter(
    x=macro_result_df.index,
    y=macro_result_df['SPY_Macro'],
    mode='lines+markers',
    line_shape='hv',
    name='SPY Macro Regime',
    line=dict(color='blue', width=2),
    marker=dict(color=['red' if m == -1 else 'green' if m == 1 else 'gray' for m in macro_result_df['SPY_Macro']])
))

fig.update_layout(
    title='Macro Regime Analysis - BTC vs SPY',
    xaxis_title='Date',
    yaxis_title='Regime (-1=Bear, 0=Neutral, 1=Bull)',
    template='plotly_dark',
    height=600,
    showlegend=True
)

fig.show()

In [ ]:
#if topo de preço atual > topo do preço anterior & dados e indicadore do topo do preço atual < dados e indicadores do topo do preço anterior == Saída forte!
#guardar a maxima do periodo e comparar com o rompimento dela em seguida e parar
#peso 2 para sharpe e sortino ratios

Graphic

In [ ]:
top_btc

[321, 324, 325]

In [ ]:
# %% [markdown]
# ## Data Visualization
# %%
fig = make_subplots(
    rows=10,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.005,
    row_heights=[4.5, 3.5, 3.5, 3.5, 3.5, 3.5, 3.5, 3.5, 3.5, 3.5]
)

# %% [markdown]
# ## Data Plotting Prices
# %%
fig.add_trace(go.Candlestick(
    x=df_stocks.index,
    open=df_stocks['Open']['BTC-USD'],
    high=df_stocks['High']['BTC-USD'],
    low=df_stocks['Low']['BTC-USD'],
    close=df_stocks['Close']['BTC-USD'],
    name='BTC-USD',

), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df_stocks.index[top_btc],
    y=df_stocks['Close']['BTC-USD'].iloc[top_btc],
    mode='markers+text',
    marker=dict(size=10,
                color='#ffffe0',
                symbol='arrow-down',),
    name=f'Top Points ({len(top_btc)} points)'),
    row=1, col=1
)

fig.add_trace(go.Scatter(
    x=df_stocks.index[top_spy],
    y=df_stocks['Close']['BTC-USD'].iloc[top_spy],
    mode='markers+text',
    marker=dict(size=10,
                color='#FFFF00',
                symbol='star',),
    name=f'Top Points ({len(top_spy)} points)'),
    row=1, col=1
)

if len(top_btc_confirmed) != 0:
    fig.add_trace(go.Scatter(
        x=df_stocks.index[top_btc_confirmed],
        y=df_stocks['Close']['BTC-USD'].iloc[top_btc_confirmed],
        mode='markers+text',
        marker=dict(size=12,
                    color='#ffffe0',
                    symbol='x',),
        name=f'Top Points (now) ({len(top_btc_confirmed)} points)'),
        row=1, col=1
    )

if len(top_spy_confirmed) != 0:
    fig.add_trace(go.Scatter(
        x=df_stocks.index[top_spy_confirmed],
        y=df_stocks['Close']['BTC-USD'].iloc[top_spy_confirmed],
        mode='markers+text',
        marker=dict(size=12,
                    color='#FFFF00',
                    symbol='x',),
        name=f'Top Points (now) ({len(top_spy_confirmed)} points)'),
        row=1, col=1
    )

fig.add_trace(go.Scatter(
    x=df_stocks.index[bottom_btc],
    y=df_stocks['Close']['BTC-USD'].iloc[bottom_btc],
    mode='markers+text',
    marker=dict(size=10,
                color='lightblue',
                symbol='arrow-up'),
    name=f'Bottom Points ({len(bottom_btc)} points)'),
    row=1, col=1
)

fig.add_trace(go.Scatter(
    x=df_stocks.index[bottom_spy],
    y=df_stocks['Close']['BTC-USD'].iloc[bottom_spy],
    mode='markers+text',
    marker=dict(size=10,
                color='#00FFFF',
                symbol='star'),
    name=f'Bottom Points ({len(bottom_spy)} points)'),
    row=1, col=1
)

if len(bottom_btc_confirmed) != 0:
    fig.add_trace(go.Scatter(
        x=df_stocks.index[bottom_btc_confirmed],
        y=df_stocks['Close']['BTC-USD'].iloc[bottom_btc_confirmed],
        mode='markers+text',
        marker=dict(size=12,
                    color='lightblue',
                    symbol='x',),
        name=f'Bottom Points (now) ({len(bottom_btc_confirmed)} points)'),
        row=1, col=1
    )

if len(bottom_spy_confirmed) != 0:
    fig.add_trace(go.Scatter(
        x=df_stocks.index[bottom_spy_confirmed],
        y=df_stocks['Close']['BTC-USD'].iloc[bottom_spy_confirmed],
        mode='markers+text',
        marker=dict(size=12,
                    color='#00FFFF',
                    symbol='x',),
        name=f'Bottom Points (now) ({len(bottom_spy_confirmed)} points)'),
        row=1, col=1
    )

###
# Plot BTC SMAs
fig.add_trace(go.Scatter(
    x=df_stocks_close_criptos.index,
    y=df_stocks_close_criptos['BTC-USD_EMA-50'],
    name='EMA-50',
    line=dict(color='lightgreen')
), row=1, col=1)


fig.add_trace(go.Scatter(
    x=df_stocks_close_criptos.index,
    y=df_stocks_close_criptos['BTC-USD_EMA-100'],
    name='EMA-100',
    line=dict(color='lightblue')
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df_stocks_close_criptos.index,
    y=df_stocks_close_criptos['BTC-USD_EMA-200'],
    name='EMA-200',
    line=dict(color='white')
), row=1, col=1)

# %% [markdown]
# ## Data Plotting MACD BTC
# %%
fig.add_trace(go.Scatter(
    x=df_stocks.index,
    y=macd,
    name='MACD',
    marker_color='lightblue',
    opacity=0.7
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=df_stocks.index,
    y=signal,
    name='Signal',
    marker_color='lightcoral',
    opacity=0.7
), row=2, col=1)
fig.add_trace(go.Bar(
    x=df_stocks.index,
    y=histogram,
    name='Histogram',
    marker_color=histogram,
), row=2, col=1)

# %% [markdown]
# ## Data Plotting MACD SPY
# %%

fig.add_trace(go.Scatter(
    x=df_stocks.index,
    y=macd_spy,
    name='MACD SPY',
    marker_color='lightblue',
    opacity=0.7
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=df_stocks.index,
    y=signal_spy,
    name='Signal SPY',
    marker_color='lightcoral',
    opacity=0.7
), row=3, col=1)
fig.add_trace(go.Bar(
    x=df_stocks.index,
    y=histogram_spy,
    name='Histogram',
    marker_color=histogram_spy,
), row=3, col=1)

# %% [markdown]
# ## Data Plotting Sharpe and Sortino Ratios BTC
# %%

fig.add_trace(go.Scatter(
    x=sharpe_ratio.index,
    y=sharpe_ratio['BTC-USD'],
    line=dict(color='white'),
    name='Sharpe Ratio BTC',
), row=4, col=1)

fig.add_trace(go.Bar(
    x=sortino_ratio.index,
    y=sortino_ratio['BTC-USD'],
    marker_color=sortino_ratio['BTC-USD'],
    name='Sortino Ratio BTC',
), row=4, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_btc_sharpe_sortino_top],
    y=sortino_ratio['BTC-USD'][mask_btc_sharpe_sortino_top],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['BTC-USD'][mask_btc_sharpe_sortino_top])*3,
                color=sortino_ratio['BTC-USD'][mask_btc_sharpe_sortino_top],
                colorscale='YlOrRd',
                symbol='diamond'),
    name='Sortino BTC > 3.5 & Sharpe > 2.0'),
    row=4, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_btc_sharpe_sortino_bottom],
    y=sortino_ratio['BTC-USD'][mask_btc_sharpe_sortino_bottom],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['BTC-USD'][mask_btc_sharpe_sortino_bottom])*6,
                color=sortino_ratio['BTC-USD'][mask_btc_sharpe_sortino_bottom],
                colorscale='BuGn',
                symbol='diamond'),
    name='Sort < -2.0 & Shar < -2.0'),
    row=4, col=1
)

fig.add_trace(go.Scatter(
    x=sharpe_ratio.index[mask_btc_sharpe_top],
    y=sharpe_ratio['BTC-USD'][mask_btc_sharpe_top],
    mode='markers',
    marker=dict(size=sharpe_ratio['BTC-USD'][mask_btc_sharpe_top]*3,
                color='lightcoral',
                symbol='circle'),
    name='Sharpe BTC > 2'),
    row=4, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_btc_sortino_top],
    y=sortino_ratio['BTC-USD'][mask_btc_sortino_top],
    mode='markers',
    marker=dict(size=sortino_ratio['BTC-USD'][mask_btc_sortino_top]*1.5,
                color=sortino_ratio['BTC-USD'][mask_btc_sortino_top],
                colorscale='PuRd',
                symbol='circle'),
    name='Sortino BTC > 3.5'),
    row=4, col=1
)

fig.add_trace(go.Scatter(
    x=sharpe_ratio.index[mask_btc_sharpe_bottom],
    y=sharpe_ratio['BTC-USD'][mask_btc_sharpe_bottom],
    mode='markers',
    marker=dict(size=abs(sharpe_ratio['BTC-USD'][mask_btc_sharpe_bottom])*3,
                color='lightgreen',
                symbol='circle'),
    name='Sharpe BTC < -2'),
    row=4, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_btc_sortino_reversion_down],
    y=sortino_ratio['BTC-USD'][mask_btc_sortino_reversion_down],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['BTC-USD'][mask_btc_sortino_reversion_down])*2,
                color=sortino_ratio['BTC-USD'][mask_btc_sortino_reversion_down],
                colorscale='YlOrRd',
                symbol='circle'),
    name=f'Sort > 3.5 & Shar < 0'),
    row=4, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_btc_sortino_bottom],
    y=sortino_ratio['BTC-USD'][mask_btc_sortino_bottom],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['BTC-USD'][mask_btc_sortino_bottom])*3,
                color='green',
                symbol='circle'),
    name=f'Sortino BTC < -2.0'),
    row=4, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_btc_sortino_reversion_up],
    y=sortino_ratio['BTC-USD'][mask_btc_sortino_reversion_up],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['BTC-USD'][mask_btc_sortino_reversion_up])*5,
                color='lightblue',
                symbol='star'),
    name='Sort < -1.5 & Shar > 0'),
    row=4, col=1
)

# %% [markdown]
# ## Data Plotting Sharpe and Sortino Ratios SPY
# %%

fig.add_trace(go.Scatter(
    x=sharpe_ratio.index,
    y=sharpe_ratio['SPY'],
    line=dict(color='white'),
    name='Sharpe Ratio SPY',
), row=5, col=1)

fig.add_trace(go.Bar(
    x=sortino_ratio.index,
    y=sortino_ratio['SPY'],
    marker_color=sortino_ratio['SPY'],
    name='Sortino Ratio SPY',
), row=5, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_spy_sharpe_sortino_top],
    y=sortino_ratio['SPY'][mask_spy_sharpe_sortino_top],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['SPY'][mask_spy_sharpe_sortino_top])*3,
                color=sortino_ratio['SPY'][mask_spy_sharpe_sortino_top],
                colorscale='YlOrRd',
                symbol='diamond'),
    name='Sortino SPY > 3.2 & Sharpe > 2.5'),
    row=5, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_spy_sharpe_sortino_bottom],
    y=sortino_ratio['SPY'][mask_spy_sharpe_sortino_bottom],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['SPY'][mask_spy_sharpe_sortino_bottom])*10,
                color=sortino_ratio['SPY'][mask_spy_sharpe_sortino_bottom],
                colorscale='BuGn',
                symbol='diamond'),
    name='Sort < 0 & Shar < -0.2'),
    row=5, col=1
)

fig.add_trace(go.Scatter(
    x=sharpe_ratio.index[mask_spy_sharpe_top],
    y=sharpe_ratio['SPY'][mask_spy_sharpe_top],
    mode='markers',
    marker=dict(size=sharpe_ratio['SPY'][mask_spy_sharpe_top]*3,
                color='lightcoral',
                symbol='circle'),
    name='Sharpe SPY > 2'),
    row=5, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_spy_sortino_top],
    y=sortino_ratio['SPY'][mask_spy_sortino_top],
    mode='markers',
    marker=dict(size=sortino_ratio['SPY'][mask_spy_sortino_top]*1.5,
                color=sortino_ratio['SPY'][mask_spy_sortino_top],
                colorscale='YlOrRd',
                symbol='circle'),
    name='Sortino SPY > 3.2'),
    row=5, col=1
)

fig.add_trace(go.Scatter(
    x=sharpe_ratio.index[mask_spy_sharpe_bottom],
    y=sharpe_ratio['SPY'][mask_spy_sharpe_bottom],
    mode='markers',
    marker=dict(size=abs(sharpe_ratio['SPY'][mask_spy_sharpe_bottom])*8,
                color='lightgreen',
                symbol='circle'),
    name='Sharpe SPY < -0.2'),
    row=5, col=1
)


fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_spy_sortino_reversion_down],
    y=sortino_ratio['SPY'][mask_spy_sortino_reversion_down],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['SPY'][mask_spy_sortino_reversion_down])*2,
                color='red',
                symbol='circle'),
    name=f'Sort > 3.5'),
    row=5, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_spy_sortino_bottom],
    y=sortino_ratio['SPY'][mask_spy_sortino_bottom],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['SPY'][mask_spy_sortino_bottom])*7,
                color='green',
                symbol='circle'),
    name=f'Sortino SPY < 0'),
    row=5, col=1
)

# %% [markdown]
# ## Data Plotting Bollinger Bands %B BTC-USD
# %%
fig.add_trace(go.Scatter(
    x=df_stocks.index,
    y=bb_percent_b,
    name='BB %B BTC',
    line=dict(color='white', width=1),
    opacity=0.7,
), row=6, col=1)

fig.update_layout(
    yaxis6=dict(
        autorange=False,
        range=[-1, 1.5],
        title="BB %B BTC"
    )
)

fig.add_hline(y=0, line_dash="dot", line_color="lightgreen", row=6, col=1)
fig.add_hline(y=0.50, line_dash="dot", line_color="gray", row=6, col=1)
fig.add_hline(y=1, line_dash="dot", line_color="lightcoral", row=6, col=1)

# %% [markdown]
# ## Data Plotting Bollinger Bands %B SPY
# %%
fig.add_trace(go.Scatter(
    x=df_stocks.index,
    y=bb_percent_spy,
    name='BB %B SPY',
    line=dict(color='white', width=1),
    opacity=0.7,
), row=7, col=1)

fig.update_layout(
    yaxis7=dict(
        autorange=False,
        range=[-1, 1.5],
        title="BB %B SPY"
    )
)

fig.add_hline(y=0, line_dash="dot", line_color="lightgreen", row=7, col=1)
fig.add_hline(y=0.50, line_dash="dot", line_color="gray", row=7, col=1)
fig.add_hline(y=1, line_dash="dot", line_color="lightcoral", row=7, col=1)

# %% [markdown]
# ## Data Plotting RSI and Stochastic BTC
# %%
fig.add_trace(go.Scatter(
            x=df_stocks.index,
            y=rsi_btc,
            name='RSI BTC',
            line=dict(color='white', width=1)
), row=8, col=1)

fig.add_trace(go.Scatter(
            x=df_stocks.index,
            y=stoch_d_btc,
            name='%D BTC',
            line=dict(color='red', width=1)
), row=8, col=1)

fig.add_hline(y=75, line_dash="dot", line_color="lightcoral", row=8, col=1)
fig.add_hline(y=50, line_dash="dot", line_color="gray", row=8, col=1)
fig.add_hline(y=25, line_dash="dot", line_color="lightgreen", row=8, col=1)


# %% [markdown]
# ## Data Plotting RSI and Stochastic SPY
# %%
fig.add_trace(go.Scatter(
            x=df_stocks.index,
            y=rsi_spy,
            name='RSI SPY',
            line=dict(color='white', width=1)
), row=9, col=1)

fig.add_trace(go.Scatter(
            x=df_stocks.index,
            y=stoch_d_spy,
            name='%D SPY',
            line=dict(color='red', width=1)
), row=9, col=1)

fig.add_hline(y=75, line_dash="dot", line_color="lightcoral", row=9, col=1)
fig.add_hline(y=50, line_dash="dot", line_color="gray", row=9, col=1)
fig.add_hline(y=25, line_dash="dot", line_color="lightgreen", row=9, col=1)


# %% [markdown]
# ## Data Plotting VIX
# %%

fig.add_trace(go.Bar(
    x=df_stocks.index,
    y=df_stocks['Close']['^VIX'],
    name='VIX',
    marker_color=df_stocks['Close']['^VIX']
), row=10, col=1)

# %% [markdown]
# ## Data Plotting End
# %%

fig.update_layout(
    title= f"Market Analysis - {df_stocks.index[0].date().strftime('%Y-%m-%d')} to {df_stocks.index[-1].date().strftime('%Y-%m-%d')}",
    title_x=0.5,
    showlegend=False,
    height=2000,
    width=1150,
    yaxis_title='Price',
    yaxis_type='log',
    yaxis2_title='MACD BTC',
    yaxis3_title='MACD SPY',
    yaxis4_title='SHP&STO BTC',
    yaxis4_type='linear',
    yaxis5_title='SHP&STO SPY',
    yaxis5_type='linear',
    yaxis6_title='BB %B BTC',
    yaxis6_type='linear',
    yaxis7_title='BB %B SPY',
    yaxis7_type='linear',
    yaxis8_title='RSI & STO BTC',
    yaxis8_type='linear',
    yaxis9_title='RSI & STO SPY',
    yaxis9_type='linear',
    yaxis10_title='VIX',
    template='plotly_dark',
    xaxis_rangeslider_visible=False,


)

fig.show()